In [1]:
!pip -q install lightgbm joblib

import os
import re
import json
import warnings
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from lightgbm import LGBMRegressor

from joblib import dump, load

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Import done.")

Import done.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/instagram_users_lifestyle.csv"

# Nếu muốn dùng toàn bộ dataset thì để MAX_ROWS = None
MAX_ROWS = 400_000


RUN_RANDOM_FOREST = True

# Số fold tạo stress_pred_feature.
OOF_FOLDS = 5

# Có dùng stress dự đoán để hỗ trợ happiness không.
# Không dùng stress thật để tránh leakage.
USE_STRESS_PRED_FEATURE = True

OUT_DIR = "/content/models"
os.makedirs(OUT_DIR, exist_ok=True)

print("Config ready.")
print("DATA_PATH:", DATA_PATH)
print("File exists:", os.path.exists(DATA_PATH))

Mounted at /content/drive
Config ready.
DATA_PATH: /content/drive/MyDrive/Colab Notebooks/instagram_users_lifestyle.csv
File exists: True


In [3]:
# 3) Load dữ liệu


if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy file tại: {DATA_PATH}\n"
        "Hãy kiểm tra lại đường dẫn hoặc tên file trong Google Drive."
    )

df = pd.read_csv(DATA_PATH)

print("Shape ban đầu:", df.shape)
display(df.head())
df.info()

Shape ban đầu: (1547896, 58)


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,...,last_login_date,average_session_length_minutes,content_type_preference,preferred_content_theme,privacy_setting_level,two_factor_auth_enabled,biometric_login_used,linked_accounts_count,subscription_status,user_engagement_score
0,1,Instagram,51,Female,India,Rural,High,Retired,Bachelor’s,Single,...,2025-11-02,5.0,Mixed,Tech,Private,Yes,No,0,Free,7.83
1,2,Instagram,64,Female,United Kingdom,Urban,Middle,Full-time employed,Other,Divorced,...,2025-03-22,14.8,Photos,Fashion,Public,No,No,3,Free,1.43
2,3,Instagram,41,Female,Canada,Urban,Middle,Student,Bachelor’s,In a relationship,...,2025-08-10,5.0,Mixed,Other,Public,Yes,Yes,1,Free,9.67
3,4,Instagram,27,Non-binary,South Korea,Urban,Middle,Unemployed,Master’s,In a relationship,...,2025-03-31,25.9,Stories,Tech,Private,No,No,1,Free,0.94
4,5,Instagram,55,Male,India,Urban,Upper-middle,Full-time employed,Bachelor’s,Single,...,2025-03-19,13.1,Videos,Food,Public,Yes,No,0,Free,1.03


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1547896 entries, 0 to 1547895
Data columns (total 58 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   user_id                         1547896 non-null  int64  
 1   app_name                        1547896 non-null  object 
 2   age                             1547896 non-null  int64  
 3   gender                          1547896 non-null  object 
 4   country                         1547896 non-null  object 
 5   urban_rural                     1547896 non-null  object 
 6   income_level                    1547896 non-null  object 
 7   employment_status               1547896 non-null  object 
 8   education_level                 1547896 non-null  object 
 9   relationship_status             1547896 non-null  object 
 10  has_children                    1547896 non-null  object 
 11  exercise_hours_per_week         1547896 non-null  float64
 12  

In [4]:
# 4) Chuẩn hóa tên cột và lấy mẫu

def clean_col_name(col):
    col = str(col).strip()
    col = col.replace("\ufeff", "")
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col

original_columns = list(df.columns)
df.columns = [clean_col_name(c) for c in df.columns]

df = df.drop_duplicates().reset_index(drop=True)

if "user_id" in df.columns:
    df = df.drop_duplicates(subset=["user_id"], keep="first").reset_index(drop=True)

original_size = len(df)

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Dữ liệu gốc sau bỏ trùng: {original_size:,} dòng")
print(f"Dữ liệu dùng để train: {len(df):,} dòng")
print("Shape:", df.shape)

Dữ liệu gốc sau bỏ trùng: 1,547,896 dòng
Dữ liệu dùng để train: 400,000 dòng
Shape: (400000, 58)


In [5]:
# 5) Xác định target

STRESS_TARGET = "perceived_stress_score"
HAPPINESS_TARGET = "self_reported_happiness"

if STRESS_TARGET not in df.columns:
    raise ValueError(f"Không tìm thấy cột {STRESS_TARGET}")

if HAPPINESS_TARGET not in df.columns:
    raise ValueError(f"Không tìm thấy cột {HAPPINESS_TARGET}")

df[STRESS_TARGET] = pd.to_numeric(df[STRESS_TARGET], errors="coerce")
df[HAPPINESS_TARGET] = pd.to_numeric(df[HAPPINESS_TARGET], errors="coerce")

df = df.dropna(subset=[STRESS_TARGET, HAPPINESS_TARGET]).reset_index(drop=True)

print("Stress target describe:")
display(df[STRESS_TARGET].describe())

print("Happiness target describe:")
display(df[HAPPINESS_TARGET].describe())

Stress target describe:


,perceived_stress_score
count,400000.000000
mean,19.969915
std,11.844684
min,0.000000
25%,10.000000
50%,20.000000
75%,30.000000
max,40.000000


Happiness target describe:


,self_reported_happiness
count,400000.000000
mean,5.504907
std,2.867938
min,1.000000
25%,3.000000
50%,6.000000
75%,8.000000
max,10.000000


In [6]:
# 6) Feature engineering

def safe_div(a, b):
    return a / b.replace(0, np.nan)

def add_if_cols_exist(data, new_col, cols, mode="sum"):
    exist_cols = [c for c in cols if c in data.columns]
    if len(exist_cols) == 0:
        return data

    if mode == "sum":
        data[new_col] = data[exist_cols].sum(axis=1)
    elif mode == "mean":
        data[new_col] = data[exist_cols].mean(axis=1)

    return data

def clip_outliers_iqr(data, cols):
    data = data.copy()
    for col in cols:
        if col not in data.columns:
            continue
        if not np.issubdtype(data[col].dtype, np.number):
            continue

        q1 = data[col].quantile(0.01)
        q99 = data[col].quantile(0.99)
        data[col] = data[col].clip(q1, q99)

    return data

def feature_engineering(data):
    data = data.copy()

    # Ép object dạng số thành numeric nếu hợp lý
    for col in data.columns:
        if data[col].dtype == "object":
            s = data[col].astype(str).str.replace(",", "", regex=False).str.strip()
            converted = pd.to_numeric(s, errors="coerce")
            if converted.notna().mean() >= 0.90:
                data[col] = converted

    # Date features
    if "last_login_date" in data.columns:
        parsed = pd.to_datetime(data["last_login_date"], errors="coerce")
        if parsed.notna().mean() > 0.5:
            max_date = parsed.max()
            data["last_login_year"] = parsed.dt.year
            data["last_login_month"] = parsed.dt.month
            data["last_login_dayofweek"] = parsed.dt.dayofweek
            data["last_login_is_weekend"] = parsed.dt.dayofweek.isin([5, 6]).astype(int)
            data["days_since_last_login"] = (max_date - parsed).dt.days
        data = data.drop(columns=["last_login_date"])

    # Account age
    if "account_creation_year" in data.columns:
        data["account_age"] = 2026 - data["account_creation_year"]

    # Basic interaction features
    if all(c in data.columns for c in ["likes_given_per_day", "comments_written_per_day", "dms_sent_per_week"]):
        data["total_interactions"] = (
            data["likes_given_per_day"]
            + data["comments_written_per_day"]
            + data["dms_sent_per_week"] / 7.0
        )

    if all(c in data.columns for c in ["dms_sent_per_week", "dms_received_per_week"]):
        data["daily_dm_total"] = (
            data["dms_sent_per_week"] + data["dms_received_per_week"]
        ) / 7.0
        data["dm_sent_received_ratio"] = safe_div(
            data["dms_sent_per_week"] + 1,
            data["dms_received_per_week"] + 1
        )

    # Usage intensity
    if all(c in data.columns for c in ["daily_active_minutes_instagram", "sessions_per_day"]):
        data["usage_intensity"] = (
            data["daily_active_minutes_instagram"] * data["sessions_per_day"]
        )
        data["minutes_per_session"] = safe_div(
            data["daily_active_minutes_instagram"],
            data["sessions_per_day"]
        )

    # Time surface total
    surface_cols = [
        "time_on_feed_per_day",
        "time_on_explore_per_day",
        "time_on_messages_per_day",
        "time_on_reels_per_day"
    ]
    existing_surface_cols = [c for c in surface_cols if c in data.columns]
    if len(existing_surface_cols) > 0:
        data["total_surface_time"] = data[existing_surface_cols].sum(axis=1)

        for col in existing_surface_cols:
            data[f"{col}_ratio"] = safe_div(data[col], data["total_surface_time"] + 1)

    # Content consumption
    content_cols = [
        "reels_watched_per_day",
        "stories_viewed_per_day",
        "posts_created_per_week"
    ]
    data = add_if_cols_exist(data, "content_activity_sum", content_cols, mode="sum")

    # Social activity
    social_cols = [
        "social_events_per_month",
        "hobbies_count",
        "books_read_per_year",
        "volunteer_hours_per_month",
        "travel_frequency_per_year"
    ]
    data = add_if_cols_exist(data, "offline_social_lifestyle_score", social_cols, mode="mean")

    # Health activity
    health_cols = [
        "exercise_hours_per_week",
        "sleep_hours_per_night",
        "daily_steps_count"
    ]
    data = add_if_cols_exist(data, "health_activity_score_raw", health_cols, mode="mean")

    # Sleep quality approximate
    if "sleep_hours_per_night" in data.columns:
        data["sleep_distance_from_8h"] = np.abs(data["sleep_hours_per_night"] - 8)
        data["sleep_under_6h"] = (data["sleep_hours_per_night"] < 6).astype(int)
        data["sleep_over_9h"] = (data["sleep_hours_per_night"] > 9).astype(int)

    # Work-life balance approximate
    if all(c in data.columns for c in ["weekly_work_hours", "sleep_hours_per_night"]):
        data["work_sleep_ratio"] = safe_div(
            data["weekly_work_hours"],
            data["sleep_hours_per_night"] * 7
        )

    if all(c in data.columns for c in ["weekly_work_hours", "exercise_hours_per_week"]):
        data["work_exercise_ratio"] = safe_div(
            data["weekly_work_hours"],
            data["exercise_hours_per_week"] + 1
        )

    # Instagram pressure features
    if all(c in data.columns for c in ["daily_active_minutes_instagram", "sleep_hours_per_night"]):
        data["usage_sleep_ratio"] = safe_div(
            data["daily_active_minutes_instagram"],
            data["sleep_hours_per_night"] * 60
        )

    if all(c in data.columns for c in ["daily_active_minutes_instagram", "exercise_hours_per_week"]):
        data["usage_exercise_ratio"] = safe_div(
            data["daily_active_minutes_instagram"],
            data["exercise_hours_per_week"] + 1
        )

    if all(c in data.columns for c in ["posts_created_per_week", "daily_active_minutes_instagram"]):
        data["posting_per_active_minute"] = safe_div(
            data["posts_created_per_week"],
            data["daily_active_minutes_instagram"] + 1
        )

    # Ads behavior
    if all(c in data.columns for c in ["ads_clicked_per_day", "ads_viewed_per_day"]):
        data["ad_click_rate"] = safe_div(
            data["ads_clicked_per_day"],
            data["ads_viewed_per_day"] + 1
        )

    # Network ratio
    if all(c in data.columns for c in ["followers_count", "following_count"]):
        data["follower_following_ratio"] = safe_div(
            data["followers_count"] + 1,
            data["following_count"] + 1
        )
        data["network_size"] = data["followers_count"] + data["following_count"]

    # Engagement per minute
    if all(c in data.columns for c in ["total_interactions", "daily_active_minutes_instagram"]):
        data["interactions_per_active_minute"] = safe_div(
            data["total_interactions"],
            data["daily_active_minutes_instagram"] + 1
        )

    # Log transforms for skewed count columns
    log_cols = [
        "followers_count",
        "following_count",
        "network_size",
        "daily_steps_count",
        "usage_intensity",
        "total_interactions",
        "content_activity_sum",
        "total_surface_time"
    ]
    for col in log_cols:
        if col in data.columns:
            data[f"log_{col}"] = np.log1p(data[col].clip(lower=0))

    # Interaction features for happiness
    if all(c in data.columns for c in ["sleep_hours_per_night", "exercise_hours_per_week"]):
        data["sleep_x_exercise"] = data["sleep_hours_per_night"] * data["exercise_hours_per_week"]

    if all(c in data.columns for c in ["daily_active_minutes_instagram", "social_events_per_month"]):
        data["usage_x_social_events"] = (
            data["daily_active_minutes_instagram"] * data["social_events_per_month"]
        )

    if all(c in data.columns for c in ["usage_intensity", "sleep_distance_from_8h"]):
        data["usage_x_bad_sleep"] = (
            data["usage_intensity"] * data["sleep_distance_from_8h"]
        )

    if all(c in data.columns for c in ["usage_intensity", "exercise_hours_per_week"]):
        data["usage_x_exercise"] = (
            data["usage_intensity"] * data["exercise_hours_per_week"]
        )

    # Clean inf
    data = data.replace([np.inf, -np.inf], np.nan)

    # Clip một số cột dễ outlier
    clip_cols = [
        "daily_active_minutes_instagram",
        "usage_intensity",
        "total_interactions",
        "followers_count",
        "following_count",
        "network_size",
        "daily_steps_count",
        "weekly_work_hours"
    ]
    data = clip_outliers_iqr(data, clip_cols)

    return data

df_fe = feature_engineering(df)

print("Shape sau feature engineering:", df_fe.shape)
display(df_fe.head())

Shape sau feature engineering: (400000, 100)


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,...,log_network_size,log_daily_steps_count,log_usage_intensity,log_total_interactions,log_content_activity_sum,log_total_surface_time,sleep_x_exercise,usage_x_social_events,usage_x_bad_sleep,usage_x_exercise
0,336337,Instagram,43,Male,Brazil,Rural,Low,Student,Some college,Single,...,6.647688,8.979543,7.501082,5.159876,5.655992,5.455321,8.06,402.0,3256.2,2351.7
1,253653,Instagram,17,Male,Germany,Urban,Low,Not employed,Some college,Single,...,7.860185,8.986697,7.349231,5.216487,5.908083,5.575949,144.80,888.0,0.0,28127.4
2,1079086,Instagram,52,Female,United Kingdom,Urban,Middle,Full-time employed,High school,Married,...,5.849325,8.964184,7.681560,5.226514,5.655992,5.575949,36.34,985.0,216.7,9968.2
3,1474133,Instagram,24,Male,Canada,Suburban,Middle,Part-time,Other,In a relationship,...,7.276556,8.992557,2.484907,3.603166,5.056246,2.564949,46.97,22.0,20.9,84.7
4,1296480,Instagram,31,Male,South Korea,Rural,High,Freelancer,Bachelor’s,Single,...,7.128496,8.993179,8.554104,5.304725,5.869297,5.598422,62.30,247.0,5187.0,46164.3


In [7]:
# 7) Chọn feature, tránh leakage

def is_bad_identifier_col(col):
    c = col.lower()

    bad_exact = {
        "user_id",
        "id",
        "name",
        "username",
        "email",
        "phone",
        "address",
        "profile_url",
        "url",
        "link"
    }

    if c in bad_exact:
        return True

    if c.endswith("_id"):
        return True

    if any(k in c for k in ["email", "phone", "username", "url", "link"]):
        return True

    return False

def get_base_features(data):
    features = []
    dropped = []

    for col in data.columns:
        if col in [STRESS_TARGET, HAPPINESS_TARGET]:
            dropped.append(col)
            continue

        if is_bad_identifier_col(col):
            dropped.append(col)
            continue

        # app_name thường chỉ có Instagram, không có ích
        if col == "app_name":
            dropped.append(col)
            continue

        # object quá nhiều giá trị thì bỏ
        if data[col].dtype == "object" and data[col].nunique() > 100:
            dropped.append(col)
            continue

        features.append(col)

    return features, dropped

BASE_FEATURES, DROPPED_COLS = get_base_features(df_fe)

def get_stress_features(base_features):
    result = []
    for c in base_features:
        cl = c.lower()
        if "stress" in cl:
            continue
        if "happiness" in cl or "happy" in cl:
            continue
        result.append(c)
    return result

def get_happiness_features(base_features):
    result = []
    for c in base_features:
        cl = c.lower()
        if "happiness" in cl or "happy" in cl:
            continue
        if "stress" in cl:
            continue
        result.append(c)
    return result

STRESS_FEATURES = get_stress_features(BASE_FEATURES)
HAPPINESS_FEATURES_BASE = get_happiness_features(BASE_FEATURES)

print("Số base features:", len(BASE_FEATURES))
print("Số stress features:", len(STRESS_FEATURES))
print("Số happiness features ban đầu:", len(HAPPINESS_FEATURES_BASE))
print("Dropped cols:", DROPPED_COLS)

Số base features: 95
Số stress features: 95
Số happiness features ban đầu: 95
Dropped cols: ['user_id', 'app_name', 'perceived_stress_score', 'self_reported_happiness', 'linked_accounts_count']


In [8]:
# 8) Hàm model và evaluate

def split_num_cat(X):
    num_cols = []
    cat_cols = []

    for c in X.columns:
        if np.issubdtype(X[c].dtype, np.number):
            num_cols.append(c)
        else:
            cat_cols.append(c)

    return num_cols, cat_cols

def build_preprocessor(X):
    num_cols, cat_cols = split_num_cat(X)

    transformers = []

    if len(num_cols) > 0:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("num", num_pipe, num_cols))

    if len(cat_cols) > 0:
        cat_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1
            ))
        ])
        transformers.append(("cat", cat_pipe, cat_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False
    )

def metrics_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

def print_metrics(title, m):
    print("\n" + title)
    print("-" * len(title))
    print(f"MAE : {m['MAE']:.4f}")
    print(f"RMSE: {m['RMSE']:.4f}")
    print(f"R2  : {m['R2']:.4f}")

def fit_eval_model(name, model, X_train, y_train, X_val, y_val, clip_range=None):
    pipe = Pipeline(steps=[
        ("preprocess", build_preprocessor(X_train)),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)

    if clip_range is not None:
        pred = np.clip(pred, clip_range[0], clip_range[1])

    m = metrics_regression(y_val, pred)

    return {
        "name": name,
        "pipeline": pipe,
        "metrics": m
    }

def get_candidate_models(task):
    models = []

    models.append((
        "HistGradientBoosting",
        HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.035,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=0.05,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=RANDOM_SEED
        )
    ))

    models.append((
        "LightGBM",
        LGBMRegressor(
            n_estimators=900,
            learning_rate=0.035,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=40,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.15,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1
        )
    ))

    if RUN_RANDOM_FOREST:
        models.append((
            "RandomForest",
            RandomForestRegressor(
                n_estimators=220,
                max_depth=28,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
        ))

    return models

print("Functions ready.")

Functions ready.


In [9]:
# 9) Train/test split

train_df, test_df = train_test_split(
    df_fe,
    test_size=0.20,
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (320000, 100)
Test shape : (80000, 100)


In [10]:
# 10) Train stress model

X_stress = train_df[STRESS_FEATURES].copy()
y_stress = train_df[STRESS_TARGET].copy()

X_stress_test = test_df[STRESS_FEATURES].copy()
y_stress_test = test_df[STRESS_TARGET].copy()

X_s_tr, X_s_val, y_s_tr, y_s_val = train_test_split(
    X_stress,
    y_stress,
    test_size=0.20,
    random_state=RANDOM_SEED
)

stress_results = []

# Baseline
dummy_stress = DummyRegressor(strategy="mean")
dummy_stress.fit(X_s_tr.select_dtypes(include=[np.number]).fillna(0), y_s_tr)
dummy_pred = dummy_stress.predict(X_s_val.select_dtypes(include=[np.number]).fillna(0))
dummy_stress_metrics = metrics_regression(y_s_val, dummy_pred)
print_metrics("Stress baseline", dummy_stress_metrics)

for name, model in get_candidate_models("stress"):
    result = fit_eval_model(
        name=name,
        model=model,
        X_train=X_s_tr,
        y_train=y_s_tr,
        X_val=X_s_val,
        y_val=y_s_val,
        clip_range=(0, 40)
    )
    stress_results.append(result)
    print_metrics(f"Stress validation - {name}", result["metrics"])

best_stress = sorted(stress_results, key=lambda x: x["metrics"]["MAE"])[0]

print("\nBest stress model:", best_stress["name"])


Stress baseline
---------------
MAE : 10.2523
RMSE: 11.8334
R2  : -0.0000

Stress validation - HistGradientBoosting
----------------------------------------
MAE : 4.8868
RMSE: 6.0964
R2  : 0.7346

Stress validation - LightGBM
----------------------------
MAE : 4.8875
RMSE: 6.1018
R2  : 0.7341

Stress validation - RandomForest
--------------------------------
MAE : 4.9290
RMSE: 6.1395
R2  : 0.7308

Best stress model: HistGradientBoosting


In [11]:
# 11) Fit stress final

stress_model_templates = dict(get_candidate_models("stress"))
best_stress_template = stress_model_templates[best_stress["name"]]

stress_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_stress)),
    ("model", clone(best_stress_template))
])

stress_pipeline.fit(X_stress, y_stress)

stress_test_pred = stress_pipeline.predict(X_stress_test)
stress_test_pred = np.clip(stress_test_pred, 0, 40)

stress_test_metrics = metrics_regression(y_stress_test, stress_test_pred)
print_metrics("STRESS TEST RESULT", stress_test_metrics)


STRESS TEST RESULT
------------------
MAE : 4.8829
RMSE: 6.0960
R2  : 0.7351


In [12]:
# 12) Tạo stress_pred_feature bằng OOF

def make_stress_oof_predictions(X, y, model_template, n_splits):
    oof = np.zeros(len(X), dtype=float)

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    fold = 1

    for tr_idx, val_idx in kf.split(X):
        print(f"OOF fold {fold}/{n_splits}")

        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_val = X.iloc[val_idx].copy()

        pipe = Pipeline(steps=[
            ("preprocess", build_preprocessor(X_tr)),
            ("model", clone(model_template))
        ])

        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_val)
        pred = np.clip(pred, 0, 40)
        oof[val_idx] = pred

        fold += 1

    return oof

if USE_STRESS_PRED_FEATURE:
    stress_oof_train = make_stress_oof_predictions(
        X=X_stress,
        y=y_stress,
        model_template=best_stress_template,
        n_splits=OOF_FOLDS
    )

    stress_pred_for_test = stress_pipeline.predict(X_stress_test)
    stress_pred_for_test = np.clip(stress_pred_for_test, 0, 40)
else:
    stress_oof_train = None
    stress_pred_for_test = None

print("Stress prediction feature ready.")

OOF fold 1/5
OOF fold 2/5
OOF fold 3/5
OOF fold 4/5
OOF fold 5/5
Stress prediction feature ready.


In [13]:
# 13) Chuẩn bị happiness dataset

X_happiness = train_df[HAPPINESS_FEATURES_BASE].copy()
y_happiness = train_df[HAPPINESS_TARGET].copy()

X_happiness_test = test_df[HAPPINESS_FEATURES_BASE].copy()
y_happiness_test = test_df[HAPPINESS_TARGET].copy()

if USE_STRESS_PRED_FEATURE:
    X_happiness["stress_pred_feature"] = stress_oof_train
    X_happiness_test["stress_pred_feature"] = stress_pred_for_test

HAPPINESS_FEATURES = list(X_happiness.columns)

print("X_happiness:", X_happiness.shape)
print("X_happiness_test:", X_happiness_test.shape)
print("Happiness features:", len(HAPPINESS_FEATURES))

X_happiness: (320000, 96)
X_happiness_test: (80000, 96)
Happiness features: 96


In [14]:
# 14) Train happiness model

X_h_tr, X_h_val, y_h_tr, y_h_val = train_test_split(
    X_happiness,
    y_happiness,
    test_size=0.20,
    random_state=RANDOM_SEED
)

happiness_results = []

# Baseline
dummy_happiness = DummyRegressor(strategy="mean")
dummy_happiness.fit(X_h_tr.select_dtypes(include=[np.number]).fillna(0), y_h_tr)
dummy_h_pred = dummy_happiness.predict(X_h_val.select_dtypes(include=[np.number]).fillna(0))
dummy_h_metrics = metrics_regression(y_h_val, dummy_h_pred)
print_metrics("Happiness baseline", dummy_h_metrics)

for name, model in get_candidate_models("happiness"):
    result = fit_eval_model(
        name=name,
        model=model,
        X_train=X_h_tr,
        y_train=y_h_tr,
        X_val=X_h_val,
        y_val=y_h_val,
        clip_range=(1, 10)
    )
    happiness_results.append(result)
    print_metrics(f"Happiness validation - {name}", result["metrics"])

best_happiness = sorted(happiness_results, key=lambda x: x["metrics"]["MAE"])[0]

print("\nBest happiness model:", best_happiness["name"])


Happiness baseline
------------------
MAE : 2.4926
RMSE: 2.8668
R2  : -0.0000

Happiness validation - HistGradientBoosting
-------------------------------------------
MAE : 2.2206
RMSE: 2.6235
R2  : 0.1625

Happiness validation - LightGBM
-------------------------------
MAE : 2.2203
RMSE: 2.6262
R2  : 0.1608

Happiness validation - RandomForest
-----------------------------------
MAE : 2.2229
RMSE: 2.6290
R2  : 0.1590

Best happiness model: LightGBM


In [15]:
# 15) Fit happiness final

happiness_model_templates = dict(get_candidate_models("happiness"))
best_happiness_template = happiness_model_templates[best_happiness["name"]]

happiness_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_happiness)),
    ("model", clone(best_happiness_template))
])

happiness_pipeline.fit(X_happiness, y_happiness)

happiness_test_pred = happiness_pipeline.predict(X_happiness_test)
happiness_test_pred = np.clip(happiness_test_pred, 1, 10)

happiness_test_metrics = metrics_regression(y_happiness_test, happiness_test_pred)

print_metrics("HAPPINESS TEST RESULT", happiness_test_metrics)


HAPPINESS TEST RESULT
---------------------
MAE : 2.2200
RMSE: 2.6239
R2  : 0.1663


In [16]:
# 16) Tổng hợp kết quả

summary_df = pd.DataFrame([
    {
        "task": "stress",
        "best_model": best_stress["name"],
        "MAE": stress_test_metrics["MAE"],
        "RMSE": stress_test_metrics["RMSE"],
        "R2": stress_test_metrics["R2"]
    },
    {
        "task": "happiness",
        "best_model": best_happiness["name"],
        "MAE": happiness_test_metrics["MAE"],
        "RMSE": happiness_test_metrics["RMSE"],
        "R2": happiness_test_metrics["R2"]
    }
])

display(summary_df)

,task,best_model,MAE,RMSE,R2
0,stress,HistGradientBoosting,4.882854,6.096040,0.735137
1,happiness,LightGBM,2.219955,2.623872,0.166304


In [17]:
# 17) Phân tích lỗi happiness

error_df = test_df[[STRESS_TARGET, HAPPINESS_TARGET]].copy()
error_df["stress_pred"] = stress_test_pred
error_df["happiness_pred"] = happiness_test_pred
error_df["stress_abs_error"] = np.abs(error_df[STRESS_TARGET] - error_df["stress_pred"])
error_df["happiness_abs_error"] = np.abs(error_df[HAPPINESS_TARGET] - error_df["happiness_pred"])

print("Thống kê lỗi:")
display(error_df[["stress_abs_error", "happiness_abs_error"]].describe())

print("Top 10 dòng happiness sai nhiều nhất:")
display(error_df.sort_values("happiness_abs_error", ascending=False).head(10))

Thống kê lỗi:


,stress_abs_error,happiness_abs_error
count,80000.000000,80000.000000
mean,4.882854,2.219955
std,3.649604,1.398759
min,0.000140,0.000116
25%,2.006000,1.043382
50%,4.140448,2.096252
75%,6.999963,3.391782
max,25.295304,7.676437


Top 10 dòng happiness sai nhiều nhất:


,perceived_stress_score,self_reported_happiness,stress_pred,happiness_pred,stress_abs_error,happiness_abs_error
3360,0,1,3.515553,8.676437,3.515553,7.676437
46685,1,1,3.207450,8.423070,2.207450,7.423070
31966,40,10,36.422443,2.658144,3.577557,7.341856
11792,0,1,4.152404,8.129273,4.152404,7.129273
48001,0,1,4.233772,8.121020,4.233772,7.121020
68782,4,1,4.456523,8.080884,0.456523,7.080884
35834,1,1,4.396644,8.048345,3.396644,7.048345
66923,1,1,4.468625,8.014392,3.468625,7.014392
58806,5,1,4.431010,8.007327,0.568990,7.007327
21423,3,1,4.604015,7.955324,1.604015,6.955324


In [18]:
import warnings
warnings.filterwarnings("ignore")

# 18) Feature importance

def get_feature_names_from_pipeline(pipe, X):
    pre = pipe.named_steps["preprocess"]
    try:
        names = pre.get_feature_names_out()
        return list(names)
    except:
        return list(X.columns)

def show_feature_importance(pipe, X, top_n=25, title="Feature Importance"):
    model = pipe.named_steps["model"]

    if not hasattr(model, "feature_importances_"):
        print(f"{title}: model không hỗ trợ feature_importances_")
        return

    feature_names = get_feature_names_from_pipeline(pipe, X)
    importances = model.feature_importances_

    n = min(len(feature_names), len(importances))

    imp = pd.DataFrame({
        "feature": feature_names[:n],
        "importance": importances[:n]
    }).sort_values("importance", ascending=False)

    print(title)
    display(imp.head(top_n))

    return imp

stress_importance = show_feature_importance(
    stress_pipeline,
    X_stress,
    top_n=25,
    title="Top stress feature importance"
)

happiness_importance = show_feature_importance(
    happiness_pipeline,
    X_happiness,
    top_n=25,
    title="Top happiness feature importance"
)

Top stress feature importance: model không hỗ trợ feature_importances_
Top happiness feature importance


,feature,importance
77,stress_pred_feature,1168
3,body_mass_index,678
62,follower_following_ratio,653
38,days_since_last_login,616
51,offline_social_lifestyle_score,590
46,time_on_feed_per_day_ratio,583
61,ad_click_rate,574
47,time_on_explore_per_day_ratio,569
11,volunteer_hours_per_month,563
48,time_on_messages_per_day_ratio,560


In [19]:
# 19) Persona clustering

CLUSTER_FEATURES_CANDIDATES = [
    "age",
    "daily_active_minutes_instagram",
    "sessions_per_day",
    "exercise_hours_per_week",
    "sleep_hours_per_night",
    "daily_steps_count",
    "weekly_work_hours",
    "social_events_per_month",
    "total_interactions",
    "usage_intensity",
    "total_surface_time",
    "offline_social_lifestyle_score",
    "usage_sleep_ratio",
    "interactions_per_active_minute",
    "stress_pred_feature"
]

available_cluster_features = [
    c for c in CLUSTER_FEATURES_CANDIDATES
    if c in X_happiness.columns
]

X_cluster = X_happiness[available_cluster_features].copy()

cluster_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_cluster_scaled = cluster_preprocess.fit_transform(X_cluster)

sample_size = min(30000, X_cluster_scaled.shape[0])
sample_idx = np.random.RandomState(RANDOM_SEED).choice(
    X_cluster_scaled.shape[0],
    size=sample_size,
    replace=False
)

scores = []

for k in range(2, 7):
    km = KMeans(
        n_clusters=k,
        random_state=RANDOM_SEED,
        n_init=10
    )
    labels = km.fit_predict(X_cluster_scaled)

    score = silhouette_score(
        X_cluster_scaled[sample_idx],
        labels[sample_idx]
    )

    scores.append({
        "k": k,
        "silhouette": score
    })

scores_df = pd.DataFrame(scores)
display(scores_df)

best_k = int(scores_df.sort_values("silhouette", ascending=False).iloc[0]["k"])

persona_model = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_SEED,
    n_init=10
)

persona_labels = persona_model.fit_predict(X_cluster_scaled)

persona_df = X_happiness.copy()
persona_df["persona"] = persona_labels
persona_df["happiness_true"] = y_happiness.values
persona_df["stress_true"] = y_stress.values

profile_cols = [
    c for c in available_cluster_features
    if c in persona_df.columns
] + ["happiness_true", "stress_true"]

persona_profile = persona_df.groupby("persona")[profile_cols].mean()

print("Best k:", best_k)
display(persona_profile)

,k,silhouette
0,2,0.247801
1,3,0.215656
2,4,0.156294
3,5,0.164791
4,6,0.147814


Best k: 2


,age,daily_active_minutes_instagram,sessions_per_day,exercise_hours_per_week,sleep_hours_per_night,daily_steps_count,weekly_work_hours,social_events_per_month,total_interactions,usage_intensity,total_surface_time,offline_social_lifestyle_score,usage_sleep_ratio,interactions_per_active_minute,stress_pred_feature,happiness_true,stress_true
persona,,,,,,,,,,,,,,,,,
0,40.928740,100.789712,5.292216,7.156710,7.037929,7999.967375,37.004077,4.003413,99.284070,690.149918,118.109692,4.799125,0.243720,1.617490,11.727680,6.215727,11.703578
1,36.846109,283.872897,15.904458,7.148952,6.963229,8000.331585,35.283856,3.998061,221.161072,4649.160714,334.477383,4.797600,0.697848,0.781895,29.087955,4.723845,29.109755


In [20]:
# 20) Đặt tên persona

global_mean = persona_df[profile_cols].mean()

def make_persona_name(row):
    parts = []

    if "daily_active_minutes_instagram" in row.index:
        if row["daily_active_minutes_instagram"] > global_mean["daily_active_minutes_instagram"]:
            parts.append("High Usage")
        else:
            parts.append("Low Usage")

    if "sleep_hours_per_night" in row.index:
        if row["sleep_hours_per_night"] < global_mean["sleep_hours_per_night"]:
            parts.append("Low Sleep")
        else:
            parts.append("Good Sleep")

    if "exercise_hours_per_week" in row.index:
        if row["exercise_hours_per_week"] > global_mean["exercise_hours_per_week"]:
            parts.append("Active")
        else:
            parts.append("Less Active")

    if "stress_true" in row.index:
        if row["stress_true"] > global_mean["stress_true"]:
            parts.append("High Stress")
        else:
            parts.append("Low Stress")

    return " - ".join(parts)

persona_names = {}

for pid, row in persona_profile.iterrows():
    persona_names[int(pid)] = make_persona_name(row)

print("Persona names:")
print(persona_names)

Persona names:
{0: 'Low Usage - Good Sleep - Active - Low Stress', 1: 'High Usage - Low Sleep - Less Active - High Stress'}


In [21]:
# 21) Save model và metadata

stress_model_path = os.path.join(OUT_DIR, "stress_pipeline.joblib")
happiness_model_path = os.path.join(OUT_DIR, "happiness_pipeline.joblib")
cluster_preprocess_path = os.path.join(OUT_DIR, "persona_preprocess.joblib")
persona_model_path = os.path.join(OUT_DIR, "persona_model.joblib")
metadata_path = os.path.join(OUT_DIR, "metadata.json")

dump(stress_pipeline, stress_model_path)
dump(happiness_pipeline, happiness_model_path)
dump(cluster_preprocess, cluster_preprocess_path)
dump(persona_model, persona_model_path)

metadata = {
    "created_at": datetime.now().isoformat(),
    "data_path": DATA_PATH,
    "max_rows": MAX_ROWS,
    "random_seed": RANDOM_SEED,
    "use_stress_pred_feature": USE_STRESS_PRED_FEATURE,
    "oof_folds": OOF_FOLDS,

    "stress_target": STRESS_TARGET,
    "happiness_target": HAPPINESS_TARGET,

    "stress_features": STRESS_FEATURES,
    "happiness_features": HAPPINESS_FEATURES,
    "cluster_features": available_cluster_features,

    "best_stress_model": best_stress["name"],
    "best_happiness_model": best_happiness["name"],

    "stress_test_metrics": stress_test_metrics,
    "happiness_test_metrics": happiness_test_metrics,

    "persona_names": persona_names,

    "dropped_cols": DROPPED_COLS,
    "original_columns": original_columns,
    "final_columns": list(df_fe.columns)
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved files:")
print(stress_model_path)
print(happiness_model_path)
print(cluster_preprocess_path)
print(persona_model_path)
print(metadata_path)

Saved files:
/content/models/stress_pipeline.joblib
/content/models/happiness_pipeline.joblib
/content/models/persona_preprocess.joblib
/content/models/persona_model.joblib
/content/models/metadata.json


In [22]:
# 22) Self-test predict

loaded_stress_pipeline = load(stress_model_path)
loaded_happiness_pipeline = load(happiness_model_path)
loaded_cluster_preprocess = load(cluster_preprocess_path)
loaded_persona_model = load(persona_model_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

def align_columns(data, expected_cols):
    data = data.copy()

    for col in expected_cols:
        if col not in data.columns:
            data[col] = np.nan

    return data[expected_cols]

def predict_from_raw_dataframe(raw_df):
    raw_df = raw_df.copy()
    raw_df.columns = [clean_col_name(c) for c in raw_df.columns]

    processed = feature_engineering(raw_df)

    X_s = align_columns(processed, loaded_metadata["stress_features"])
    stress_pred = loaded_stress_pipeline.predict(X_s)
    stress_pred = np.clip(stress_pred, 0, 40)

    happiness_features = loaded_metadata["happiness_features"]

    h_base_features = [
        c for c in happiness_features
        if c != "stress_pred_feature"
    ]

    X_h = align_columns(processed, h_base_features)

    if "stress_pred_feature" in happiness_features:
        X_h["stress_pred_feature"] = stress_pred

    X_h = align_columns(X_h, happiness_features)

    happiness_pred = loaded_happiness_pipeline.predict(X_h)
    happiness_pred = np.clip(happiness_pred, 1, 10)

    cluster_features = loaded_metadata["cluster_features"]
    X_c = align_columns(X_h, cluster_features)
    X_c_scaled = loaded_cluster_preprocess.transform(X_c)
    persona_id = loaded_persona_model.predict(X_c_scaled)

    persona_name = [
        loaded_metadata["persona_names"].get(str(int(pid)), f"Persona_{int(pid)}")
        for pid in persona_id
    ]

    result = pd.DataFrame({
        "stress_pred": stress_pred,
        "happiness_pred": happiness_pred,
        "persona_id": persona_id,
        "persona_name": persona_name
    })

    return result

raw_sample = test_df.drop(columns=[STRESS_TARGET, HAPPINESS_TARGET], errors="ignore").head(10)
pred_sample = predict_from_raw_dataframe(raw_sample)

print("Prediction sample:")
display(pred_sample)

print("Actual sample:")
display(test_df[[STRESS_TARGET, HAPPINESS_TARGET]].head(10))

Prediction sample:


,stress_pred,happiness_pred,persona_id,persona_name
0,35.680643,2.873138,1,High Usage - Low Sleep - Less Active - High St...
1,12.531927,5.861487,0,Low Usage - Good Sleep - Active - Low Stress
2,22.692897,5.344922,0,Low Usage - Good Sleep - Active - Low Stress
3,33.182253,4.169134,1,High Usage - Low Sleep - Less Active - High St...
4,5.988589,7.192613,0,Low Usage - Good Sleep - Active - Low Stress
5,28.597785,5.296847,1,High Usage - Low Sleep - Less Active - High St...
6,12.927722,5.705075,0,Low Usage - Good Sleep - Active - Low Stress
7,27.175775,5.450098,1,High Usage - Low Sleep - Less Active - High St...
8,26.712416,5.435444,1,High Usage - Low Sleep - Less Active - High St...
9,20.387639,5.482147,0,Low Usage - Good Sleep - Active - Low Stress


Actual sample:


,perceived_stress_score,self_reported_happiness
0,38,7
1,12,8
2,16,3
3,31,2
4,9,9
5,32,8
6,14,4
7,40,10
8,17,3
9,20,5
